# Wallet PnL Chart

Fetches cumulative PnL time-series from the Polymarket user-pnl API for a list of wallets
and plots them on a single interactive chart.

In [9]:
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import requests

## Parameters

In [ ]:
WALLETS = ['0xcceb22d524e186153cffe79f13c0aeb75889f030']

# interval: 'all' | '1d' | '1w' | '1m' | '6m' | '1y'
INTERVAL = "all"

# fidelity: '1h' | '1d'
    FIDELITY = "1"

# shorten labels on the chart (shows first 6 + last 4 chars of address)
SHORTEN_LABELS = True

## 1 · Fetch PnL series

In [11]:
API_URL = "https://user-pnl-api.polymarket.com/user-pnl"


def fetch_pnl(wallet: str, interval: str = "all", fidelity: str = "1d") -> pd.DataFrame:
    """Fetch cumulative PnL series for a single wallet.

    Returns a DataFrame with columns: dt (UTC datetime), pnl (float).
    Returns an empty DataFrame if the request fails or returns no data.
    """
    resp = requests.get(
        API_URL,
        params={"user_address": wallet, "interval": interval, "fidelity": fidelity},
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame(columns=["dt", "pnl"])
    df = pd.DataFrame(data).rename(columns={"t": "dt", "p": "pnl"})
    df["dt"] = pd.to_datetime(df["dt"], unit="s", utc=True)
    df["pnl"] = df["pnl"].astype(float)
    return df.sort_values("dt").reset_index(drop=True)


def label(wallet: str) -> str:
    return (wallet[:6] + "..." + wallet[-4:]) if SHORTEN_LABELS else wallet


series: dict[str, pd.DataFrame] = {}
for wallet in WALLETS:
    try:
        df = fetch_pnl(wallet, INTERVAL, FIDELITY)
        series[wallet] = df
        print(f"{label(wallet)}  →  {len(df):4d} data points  "
              f"last pnl: {df['pnl'].iloc[-1]:,.2f} USDC" if len(df) else f"{label(wallet)}  →  no data")
    except Exception as e:
        print(f"{label(wallet)}  →  ERROR: {e}")
        series[wallet] = pd.DataFrame(columns=["dt", "pnl"])

0xcceb...f030  →  11157 data points  last pnl: -15,453.41 USDC


## 2 · Plot

In [12]:
fig = go.Figure()

for wallet, df in series.items():
    if df.empty:
        continue
    fig.add_trace(
        go.Scatter(
            x=df["dt"],
            y=df["pnl"],
            mode="lines",
            name=label(wallet),
            hovertemplate="%{x|%Y-%m-%d}<br>PnL: %{y:,.2f} USDC<extra>" + label(wallet) + "</extra>",
        )
    )

fig.add_hline(y=0, line_dash="dot", line_color="gray", opacity=0.5)

fig.update_layout(
    title=f"Cumulative PnL by Wallet (interval={INTERVAL}, fidelity={FIDELITY})",
    xaxis_title="Date",
    yaxis_title="Cumulative PnL (USDC)",
    legend_title="Wallet",
    hovermode="x unified",
    template="plotly_white",
)

fig.show(renderer="browser")